# Phenotype Data Integration Pipeline
This notebook integrates phenotype data from multiple sources (SPARK dataset) with configurable feature names for flexible testing on new datasets.

## Section 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## Section 2: Load and Configure Feature Names
Define configurable feature names as dictionaries. Modify these to match your dataset structure.

In [ ]:
# ========== CONFIGURATION SECTION - MODIFY THESE FOR NEW DATASETS ==========

# Data directory path
BASE_PHENO_DIR = '../SPARK_collection_v9_2022-12-12'
OUTPUT_DIR = '../PhenotypeClasses/data'

# Age filtering parameters
MIN_AGE_YEARS = 4
MAX_AGE_YEARS = 18
MAX_MISSING_VALUES_THRESHOLD = 1
MISSING_VALUE_PERCENTAGE_THRESHOLD = 0.1  # Drop features with > 10% missing

# Subject identifier column name
SUBJECT_ID_COL = 'subject_sp_id'

# ========== FEATURE CONFIGURATION DICTIONARIES ==========
# These map logical feature names to actual column names in your datasets
# Modify these dictionaries to match your dataset structure

# SCQ (Social Communication Questionnaire) features to DROP
SCQ_DROP_COLS = [
    'respondent_sp_id', 'family_sf_id', 'biomother_sp_id', 
    'biofather_sp_id', 'current_depend_adult', 'age_at_eval_months',
    'scq_measure_validity_flag', 'eval_year', 'missing_values',
    'summary_score', 'q01_phrases'
]

# Background History features to DROP
BH_DROP_COLS = [
    'respondent_sp_id', 'family_sf_id', 'biomother_sp_id', 
    'biofather_sp_id', 'sex', 'current_depend_adult', 
    'age_at_eval_months', 'age_at_eval_years'
]

BH_FINAL_DROP_COLS = [
    'hand', 'survey_version', 'eval_year', 'gender', 'child_lives_with', 
    'child_lives_with_v2', 'mother_highest_education', 
    'father_highest_education', 'annual_household_income', 'zygosity', 
    'twin_asd', 'twin_partic', 'twin_mult_birth', 'bghx_validity_flag', 
    'child_grade_school', 'sped_y_n'
]

# RBS-R (Repetitive Behavior Scale-Revised) features to DROP
RBSR_DROP_COLS = [
    'respondent_sp_id', 'family_sf_id', 'biomother_sp_id', 
    'biofather_sp_id', 'sex', 'asd', 'current_depend_adult', 
    'age_at_eval_months', 'age_at_eval_years', 'rbsr_validity_flag', 
    'overall_score', 'overall_number_items', 'total_final_score', 
    'missing_values', 'eval_year'
]

# CBCL 6-18 (Child Behavior Checklist) features to DROP
CBCL_DROP_COLS = [
    'respondent_sp_id', 'family_sf_id', 'biomother_sp_id', 
    'biofather_sp_id', 'sex', 'current_depend_adult', 'asd', 
    'age_at_eval_months', 'age_at_eval_years', 'cbcl_validity_flag', 
    'q056_h_other', 'anxious_depressed_raw_score', 
    'anxious_depressed_percentile', 'anxious_depressed_range', 
    'withdrawn_depressed_raw_score', 'withdrawn_depressed_percentile', 
    'withdrawn_depressed_range', 'somatic_complaints_raw_score', 
    'somatic_complaints_percentile', 'somatic_complaints_range', 
    'social_problems_raw_score', 'social_problems_percentile', 
    'social_problems_range', 'thought_problems_raw_score', 
    'thought_problems_percentile', 'thought_problems_range', 
    'attention_problems_raw_score', 'attention_problems_percentile', 
    'attention_problems_range', 'rule_breaking_behavior_raw_score', 
    'rule_breaking_behavior_percentile', 'rule_breaking_behavior_range', 
    'aggressive_behavior_raw_score', 'aggressive_behavior_percentile', 
    'aggressive_behavior_range', 'internalizing_problems_raw_score', 
    'internalizing_problems_percentile', 'internalizing_problems_range', 
    'externalizing_problems_raw_score', 'externalizing_problems_percentile', 
    'externalizing_problems_range', 'total_problems_raw_score', 
    'total_problems_percentile', 'total_problems_range', 
    'other_problems_raw_score', 'obsessive_compulsive_problems_raw_score', 
    'obsessive_compulsive_problems_percentile', 
    'obsessive_compulsive_problems_range', 'sluggish_cognitive_tempo_raw_score', 
    'sluggish_cognitive_tempo_percentile', 'sluggish_cognitive_tempo_range', 
    'stress_problems_raw_score', 'stress_problems_percentile', 
    'stress_problems_range', 'dsm5_conduct_problems_raw_score', 
    'dsm5_conduct_problems_percentile', 'dsm5_conduct_problems_range', 
    'dsm5_somatic_problems_raw_score', 'dsm5_somatic_problems_percentile', 
    'dsm5_somatic_problems_range', 'dsm5_oppositional_defiant_raw_score', 
    'dsm5_oppositional_defiant_percentile', 'dsm5_oppositional_defiant_range', 
    'dsm5_attention_deficit_hyperactivity_raw_score', 
    'dsm5_attention_deficit_hyperactivity_percentile', 
    'dsm5_attention_deficit_hyperactivity_range', 'dsm5_anxiety_problems_raw_score', 
    'dsm5_anxiety_problems_percentile', 'dsm5_anxiety_problems_range', 
    'dsm5_depressive_problems_raw_score', 'dsm5_depressive_problems_percentile', 
    'dsm5_depressive_problems_range', 'reading_eng_language', 
    'history_social_studies', 'arithmetic_math', 'science'
]

# Categorical value mappings for CBCL
CBCL_VALUE_MAPPINGS = {
    'above_average': 0, 'average': 1, 'below_average': 2, 'failing': 3,
    'none': 3, '1': 2, '2_3': 1, '4_more': 0,
    'less_1': 2, '1_2': 1, '3_more': 0,
    'worse': 2, 'average': 1, 'better': 0,
    'has no brothers or sisters': 1
}

# Sex encoding
SEX_ENCODING = {'Male': 1, 'Female': 0}

print("Configuration loaded successfully!")
print(f"Base directory: {BASE_PHENO_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Age range: {MIN_AGE_YEARS}-{MAX_AGE_YEARS} years")

## Section 3: Load Phenotype Data

In [ ]:
### Load SCQ (Social Communication Questionnaire) Data
print("Loading SCQ data...")
scqdf = pd.read_csv(f'{BASE_PHENO_DIR}/scq_2022-12-12.csv', header=0, index_col=None)
print(f"SCQ raw shape: {scqdf.shape}")
print(f"SCQ columns: {scqdf.columns.tolist()[:10]}...")  # Show first 10 columns
print(f"\nSCQ data info:\n{scqdf.info()}")

In [ ]:
### Load Background History Data
print("\nLoading Background History data...")
bhcdf = pd.read_csv(f'{BASE_PHENO_DIR}/background_history_child_2022-12-12.csv')
bhsdf = pd.read_csv(f'{BASE_PHENO_DIR}/background_history_sibling_2022-12-12.csv')
print(f"Background History Child raw shape: {bhcdf.shape}")
print(f"Background History Sibling raw shape: {bhsdf.shape}")

In [ ]:
### Load RBS-R (Repetitive Behavior Scale-Revised) Data
print("\nLoading RBS-R data...")
rbsr = pd.read_csv(f'{BASE_PHENO_DIR}/rbsr_2022-12-12.csv')
print(f"RBS-R raw shape: {rbsr.shape}")

### Load CBCL 6-18 (Child Behavior Checklist) Data
print("\nLoading CBCL 6-18 data...")
cbcl_2 = pd.read_csv(f'{BASE_PHENO_DIR}/cbcl_6_18_2022-12-12.csv')
print(f"CBCL 6-18 raw shape: {cbcl_2.shape}")

## Section 4: Data Exploration and Validation

In [ ]:
### Validate SCQ data
print("=" * 60)
print("SCQ Data Validation")
print("=" * 60)
print(f"Total records: {scqdf.shape[0]}")
print(f"Age range in data: {scqdf['age_at_eval_years'].min()} - {scqdf['age_at_eval_years'].max()}")
print(f"Missing values distribution:\n{scqdf['missing_values'].value_counts()}")
print(f"\nMissing data per column:\n{scqdf.isnull().sum()}")

# Check if required columns exist
required_scq_cols = [SUBJECT_ID_COL, 'age_at_eval_years', 'missing_values', 'sex']
missing_cols = [col for col in required_scq_cols if col not in scqdf.columns]
if missing_cols:
    print(f"\n⚠️  WARNING: Missing columns in SCQ: {missing_cols}")
else:
    print(f"\n✓ All required SCQ columns present")

In [ ]:
### Validate Background History data
print("\n" + "=" * 60)
print("Background History Data Validation")
print("=" * 60)
print(f"Child records: {bhcdf.shape[0]}, Sibling records: {bhsdf.shape[0]}")
if 'age_at_eval_years' in bhcdf.columns:
    print(f"Age range in child data: {bhcdf['age_at_eval_years'].min()} - {bhcdf['age_at_eval_years'].max()}")
if 'age_at_eval_years' in bhsdf.columns:
    print(f"Age range in sibling data: {bhsdf['age_at_eval_years'].min()} - {bhsdf['age_at_eval_years'].max()}")

### Validate RBS-R data
print("\n" + "=" * 60)
print("RBS-R Data Validation")
print("=" * 60)
print(f"Total records: {rbsr.shape[0]}")
if 'age_at_eval_years' in rbsr.columns:
    print(f"Age range: {rbsr['age_at_eval_years'].min()} - {rbsr['age_at_eval_years'].max()}")
if 'missing_values' in rbsr.columns:
    print(f"Missing values distribution:\n{rbsr['missing_values'].value_counts()}")

### Validate CBCL data
print("\n" + "=" * 60)
print("CBCL 6-18 Data Validation")
print("=" * 60)
print(f"Total records: {cbcl_2.shape[0]}")
print(f"Total columns: {cbcl_2.shape[1]}")
print(f"\nMissing data per column (top 10):\n{cbcl_2.isnull().sum().head(10)}")

## Section 5: Feature Engineering and Transformation

In [ ]:
### Process SCQ Data
print("Processing SCQ data...")
# Filter by age and missing values
scqdf_filtered = scqdf.loc[
    (scqdf['age_at_eval_years'] <= MAX_AGE_YEARS) & 
    (scqdf['missing_values'] < MAX_MISSING_VALUES_THRESHOLD) & 
    (scqdf['age_at_eval_years'] >= MIN_AGE_YEARS)
]
print(f"After age/missing filter: {scqdf_filtered.shape[0]} records")

# Set index and drop columns
scqdf_processed = scqdf_filtered.set_index(SUBJECT_ID_COL, drop=True).drop(
    columns=SCQ_DROP_COLS, errors='ignore'
)

# Encode sex
if 'sex' in scqdf_filtered.columns:
    scqdf_processed['sex'] = scqdf_filtered['sex'].map(SEX_ENCODING)
    scqdf_processed['sex'] = scqdf_processed['sex'].astype(int)

print(f"SCQ processed shape: {scqdf_processed.shape}")
print(f"SCQ sample:\n{scqdf_processed.head()}")

In [ ]:
### Process Background History Data
print("\nProcessing Background History data...")
# Filter by age
bhcdf_filtered = bhcdf.loc[
    (bhcdf['age_at_eval_years'] <= MAX_AGE_YEARS) & 
    (bhcdf['age_at_eval_years'] >= MIN_AGE_YEARS)
]
bhsdf_filtered = bhsdf.loc[
    (bhsdf['age_at_eval_years'] <= MAX_AGE_YEARS) & 
    (bhsdf['age_at_eval_years'] >= MIN_AGE_YEARS)
]

# Set index and drop columns
bhcdf_processed = bhcdf_filtered.set_index(SUBJECT_ID_COL, drop=True).drop(
    columns=BH_DROP_COLS, errors='ignore'
)
bhsdf_processed = bhsdf_filtered.set_index(SUBJECT_ID_COL, drop=True).drop(
    columns=BH_DROP_COLS, errors='ignore'
)

# Combine child and sibling data
bhdf = pd.concat([bhcdf_processed, bhsdf_processed], join='inner')
bhdf = bhdf.drop(columns=BH_FINAL_DROP_COLS, errors='ignore')
bhdf = bhdf[~bhdf.index.duplicated(keep=False)]

print(f"Background History combined shape: {bhdf.shape}")
print(f"Background History sample:\n{bhdf.head()}")

In [ ]:
### Process RBS-R Data
print("\nProcessing RBS-R data...")
rbsr_filtered = rbsr.loc[
    (rbsr['age_at_eval_years'] <= MAX_AGE_YEARS) & 
    (rbsr['missing_values'] < MAX_MISSING_VALUES_THRESHOLD) & 
    (rbsr['age_at_eval_years'] >= MIN_AGE_YEARS)
]

rbsr_processed = rbsr_filtered.set_index(SUBJECT_ID_COL, drop=True).drop(
    columns=RBSR_DROP_COLS, errors='ignore'
)

print(f"RBS-R processed shape: {rbsr_processed.shape}")
print(f"RBS-R sample:\n{rbsr_processed.head()}")

In [ ]:
### Process CBCL 6-18 Data
print("\nProcessing CBCL 6-18 data...")
cbcl_processed = cbcl_2.set_index(SUBJECT_ID_COL, drop=True).drop(
    columns=CBCL_DROP_COLS, errors='ignore'
)

# Convert categorical variables to numerical
cbcl_processed = cbcl_processed.replace(CBCL_VALUE_MAPPINGS)
cbcl_processed = cbcl_processed[~cbcl_processed.index.duplicated(keep=False)]

print(f"CBCL processed shape: {cbcl_processed.shape}")
print(f"CBCL sample:\n{cbcl_processed.head()}")

## Section 6: Data Integration and Merging

In [ ]:
### Integrate all phenotype measures
print("Integrating all phenotype datasets...")
print("\nDataset shapes before merging:")
print(f"  SCQ: {scqdf_processed.shape}")
print(f"  Background History: {bhdf.shape}")
print(f"  RBS-R: {rbsr_processed.shape}")
print(f"  CBCL: {cbcl_processed.shape}")

# Merge all datasets using inner join (keep only samples with all measures)
finaldf = pd.concat(
    [scqdf_processed, bhdf, rbsr_processed, cbcl_processed], 
    axis=1, 
    join='inner'
)

# Remove duplicate columns
finaldf = finaldf.loc[:, ~finaldf.columns.duplicated()]

print(f"\nFinal integrated shape: {finaldf.shape}")
print(f"Number of subjects: {finaldf.shape[0]}")
print(f"Number of features: {finaldf.shape[1]}")
print(f"\nFeature names (first 20):\n{finaldf.columns.tolist()[:20]}")

## Section 7: Quality Checks and Filtering

In [ ]:
### Quality control - check missing values
print("Quality Control Analysis")
print("=" * 60)

# Calculate missing value percentages
missing_pct = (finaldf.isna().sum() / finaldf.shape[0] * 100).sort_values(ascending=False)
print(f"Features with missing values (top 10):")
print(missing_pct.head(10))

# Drop features with excessive missing values
print(f"\nDropping features with >{MISSING_VALUE_PERCENTAGE_THRESHOLD*100:.0f}% missing values...")
missing_mask = (finaldf.isna().sum() / finaldf.shape[0]) < MISSING_VALUE_PERCENTAGE_THRESHOLD
num_dropped = (~missing_mask).sum()
finaldf_filtered = finaldf.loc[:, missing_mask]

print(f"Dropped {num_dropped} features with excessive missing values")
print(f"Remaining features: {finaldf_filtered.shape[1]}")

# Remove rows with any remaining missing values
print(f"\nRemoving rows with missing values...")
initial_samples = finaldf_filtered.shape[0]
clean_df = finaldf_filtered.dropna(axis=0)
removed_samples = initial_samples - clean_df.shape[0]

print(f"Removed {removed_samples} rows with missing values")
print(f"Final clean dataset shape: {clean_df.shape}")
print(f"Final dataset (sample):\n{clean_df.head()}")

In [ ]:
### Summary statistics
print("\nFinal Dataset Summary Statistics")
print("=" * 60)
print(f"Shape: {clean_df.shape}")
print(f"\nData types:\n{clean_df.dtypes.value_counts()}")

# Check sex distribution if available
if 'sex' in clean_df.columns:
    print(f"\nSex distribution:")
    print(clean_df['sex'].value_counts())

# Basic statistics
print(f"\nBasic statistics (first 5 features):")
print(clean_df.iloc[:, :5].describe())

## Section 8: Export Processed Data

In [ ]:
### Save processed data
print("Exporting processed data...")
print("=" * 60)

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define output filename
output_filename = 'spark_5392_cohort.txt'
output_path = os.path.join(OUTPUT_DIR, output_filename)

# Save to TSV format
clean_df.to_csv(output_path, sep='\t')
print(f"✓ Data saved to: {output_path}")
print(f"  Shape: {clean_df.shape}")
print(f"  File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")

# Optional: Also save as CSV
csv_path = os.path.join(OUTPUT_DIR, 'spark_5392_cohort.csv')
clean_df.to_csv(csv_path)
print(f"\n✓ Data also saved to: {csv_path}")

# Optional: Save data dictionary
print("\nData Summary:")
print(f"  Total samples: {clean_df.shape[0]}")
print(f"  Total features: {clean_df.shape[1]}")
print(f"  Features: {list(clean_df.columns)}")

## Tips for Testing New Datasets

1. **Modify Configuration Section**: Update the paths, column names, and filtering parameters in cell 3 to match your dataset structure.

2. **Update Feature Names**: Edit the `*_DROP_COLS` dictionaries to match the columns in your dataset files.

3. **Adjust Value Mappings**: Modify `CBCL_VALUE_MAPPINGS` and `SEX_ENCODING` if your categorical encodings differ.

4. **Run Cell by Cell**: Execute cells sequentially to debug any issues with your dataset.

5. **Monitor Quality Checks**: Review Section 7 outputs to understand your data quality and what's being filtered.

6. **Export Options**: Modify the export format (TSV/CSV) and filename in cell 13 as needed.